In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader


In [3]:
class SAROilSpillDataset(Dataset):
    def __init__(self, base_dir, split):
        """
        base_dir: /content/drive/MyDrive/oil_spill_project/data/sentinel/tiles
        split: train | val | primary_test
        """
        self.images_dir = os.path.join(base_dir, split, "images")
        self.masks_dir = os.path.join(base_dir, split, "masks")

        # IMPORTANT: iterate ONLY over masks
        self.mask_files = sorted([
            f for f in os.listdir(self.masks_dir) if f.endswith(".png")
        ])

    def __len__(self):
        return len(self.mask_files)

    def __getitem__(self, idx):
        mask_name = self.mask_files[idx]

        img_path = os.path.join(self.images_dir, mask_name)
        mask_path = os.path.join(self.masks_dir, mask_name)

        # Read image (SAR grayscale)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None or mask is None:
            raise RuntimeError(f"Failed to read {mask_name}")

        # Normalize SAR image → [0,1]
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)

        # Binary mask (already 0/1, but enforce)
        mask = (mask > 0).astype(np.float32)

        # Add channel dimension
        img = torch.from_numpy(img).unsqueeze(0)      # [1, H, W]
        mask = torch.from_numpy(mask).unsqueeze(0)    # [1, H, W]

        return img, mask


In [4]:
BASE_DIR = "/content/drive/MyDrive/oil_spill_project/data/sentinel/tiles"

train_dataset = SAROilSpillDataset(BASE_DIR, "train")
val_dataset   = SAROilSpillDataset(BASE_DIR, "val")
test_dataset  = SAROilSpillDataset(BASE_DIR, "primary_test")

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


Train: 246
Val: 76
Test: 75


In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,   # IMPORTANT for evaluation
    shuffle=False
)


In [6]:
imgs, masks = next(iter(train_loader))

print("Image batch:", imgs.shape)
print("Mask batch :", masks.shape)
print("Image min/max:", imgs.min().item(), imgs.max().item())
print("Unique mask values:", torch.unique(masks))


Image batch: torch.Size([8, 1, 400, 400])
Mask batch : torch.Size([8, 1, 400, 400])
Image min/max: 0.0 1.0
Unique mask values: tensor([0., 1.])


In [7]:
import torch
import torch.nn as nn


In [8]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


In [9]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # Output
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


In [10]:
model = UNet(in_channels=1, out_channels=1)
print(model)


UNet(
  (enc1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (block): Sequential(
      (0): Conv2d(128, 256, kernel_si

In [11]:
x = torch.randn(1, 1, 400, 400)   # dummy SAR image
y = model(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)


Input shape : torch.Size([1, 1, 400, 400])
Output shape: torch.Size([1, 1, 400, 400])


In [12]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        """
        preds: raw logits (before sigmoid) [B,1,H,W]
        targets: binary masks {0,1}         [B,1,H,W]
        """
        preds = torch.sigmoid(preds)

        preds = preds.view(preds.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (preds * targets).sum(dim=1)
        dice = (2. * intersection + self.smooth) / (
            preds.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )

        return 1 - dice.mean()


In [13]:
bce_loss = nn.BCEWithLogitsLoss()


In [14]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, preds, targets):
        loss_bce = self.bce(preds, targets)
        loss_dice = self.dice(preds, targets)
        return self.bce_weight * loss_bce + self.dice_weight * loss_dice


In [15]:
criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)


In [16]:
# fake batch
preds = torch.randn(4, 1, 400, 400)   # logits
targets = torch.randint(0, 2, (4, 1, 400, 400)).float()

loss = criterion(preds, targets)
print("Loss value:", loss.item())


Loss value: 0.6540848016738892


In [17]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [18]:
model = UNet(in_channels=1, out_channels=1).to(device)
criterion = BCEDiceLoss().to(device)


In [19]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)


In [21]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)
        running_loss += loss.item()

    return running_loss / len(loader)


In [22]:
num_epochs = 20
best_val_loss = float("inf")

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss = validate(model, val_loader, criterion)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_unet_oilspill.pth")
        print("✔ Best model saved")


Epoch [1/20] | Train Loss: 0.8016 | Val Loss: 0.7313
✔ Best model saved
Epoch [2/20] | Train Loss: 0.6677 | Val Loss: 0.6408
✔ Best model saved
Epoch [3/20] | Train Loss: 0.6395 | Val Loss: 0.7525
Epoch [4/20] | Train Loss: 0.6240 | Val Loss: 0.6146
✔ Best model saved
Epoch [5/20] | Train Loss: 0.6099 | Val Loss: 0.5941
✔ Best model saved
Epoch [6/20] | Train Loss: 0.5973 | Val Loss: 0.5994
Epoch [7/20] | Train Loss: 0.5883 | Val Loss: 0.5736
✔ Best model saved
Epoch [8/20] | Train Loss: 0.5803 | Val Loss: 0.5883
Epoch [9/20] | Train Loss: 0.5730 | Val Loss: 0.5511
✔ Best model saved
Epoch [10/20] | Train Loss: 0.5641 | Val Loss: 0.5682
Epoch [11/20] | Train Loss: 0.5518 | Val Loss: 0.5413
✔ Best model saved
Epoch [12/20] | Train Loss: 0.5493 | Val Loss: 0.5302
✔ Best model saved
Epoch [13/20] | Train Loss: 0.5378 | Val Loss: 0.5406
Epoch [14/20] | Train Loss: 0.5372 | Val Loss: 0.5095
✔ Best model saved
Epoch [15/20] | Train Loss: 0.5256 | Val Loss: 0.5094
✔ Best model saved
Epoch [16